# 01 · Data first look

What Databento's `ohlcv-1d` parent-symbology pull actually contains, before any cleaning. Each observation below motivated a step in `futures_lab.data.load`.

In [1]:
import pandas as pd

from futures_lab.config import ROOTS
from futures_lab.data.contracts import outright_pattern
from futures_lab.data.load import load_raw

raw = {root: load_raw(root) for root in ROOTS}
es = raw["ES"]
es.head()

,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol
ts_event,,,,,,,,,
2016-01-03 00:00:00+00:00,35,1,49705,2037.75,2043.5,2037.5,2040.25,10915,ESH6
2016-01-03 00:00:00+00:00,35,1,2615,2024.50,2029.5,2024.5,2027.75,201,ESU6
2016-01-03 00:00:00+00:00,35,1,6505,2032.50,2036.0,2032.5,2032.50,16,ESM6
2016-01-03 00:00:00+00:00,35,1,10088,-13.20,-13.2,-13.3,-13.30,3,ESH6-ESU6
2016-01-04 00:00:00+00:00,35,1,10123,-7.00,-6.3,-7.0,-6.30,105,ESM6-ESU6


## Shape

The index is `ts_event` in UTC and is **not unique**: every instrument trading that day shares the timestamp. Label-based selection on it (`df.loc[idx]`) returns all of them, which is how the first version of the volume-leader function returned every row.

In [2]:
es.info()
print("index unique:", es.index.is_unique)

<class 'pandas.DataFrame'>
DatetimeIndex: 20206 entries, 2016-01-03 00:00:00+00:00 to 2026-06-30 00:00:00+00:00
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   rtype          20206 non-null  uint8  
 1   publisher_id   20206 non-null  uint16 
 2   instrument_id  20206 non-null  uint32 
 3   open           20206 non-null  float64
 4   high           20206 non-null  float64
 5   low            20206 non-null  float64
 6   close          20206 non-null  float64
 7   volume         20206 non-null  uint64 
 8   symbol         20206 non-null  str    
dtypes: float64(4), str(1), uint16(1), uint32(1), uint64(1), uint8(1)
memory usage: 1.3 MB
index unique: False


## Symbol mix

Parent symbology returns outrights *and* calendar spreads, butterflies, crack spreads and user-defined strategies. Spread prices are differences and can be negative. More outright `instrument_id`s than outright symbol strings means some strings name two different contracts.

In [3]:
rows = []
for root, df in raw.items():
    is_out = df["symbol"].str.fullmatch(outright_pattern(root).pattern)
    rows.append(
        {
            "root": root,
            "rows": len(df),
            "symbols": df["symbol"].nunique(),
            "outright symbols": df.loc[is_out, "symbol"].nunique(),
            "outright instrument_ids": df.loc[is_out, "instrument_id"].nunique(),
            "non-outright rows": int((~is_out).sum()),
            "min close, all rows": df["close"].min(),
        }
    )
pd.DataFrame(rows).set_index("root")

,rows,symbols,outright symbols,outright instrument_ids,non-outright rows,"min close, all rows"
root,,,,,,
ES,20206,187,40,45,9130,-49.950000
ZB,10350,343,40,44,3422,-2.148438
CL,593497,3499,136,179,530971,-62.300000
GC,95181,1219,120,153,63928,-696.400000
6E,38091,600,118,126,18633,-0.000440


In [4]:
# A sample of what is not an outright
sorted({s for df in raw.values() for s in df["symbol"].unique() if "-" in s or ":" in s})[:12]

['6EF0-6EV9',
 '6EF0-6EX9',
 '6EF0-6EZ9',
 '6EF1-6EV0',
 '6EF1-6EX0',
 '6EF1-6EZ0',
 '6EF2-6EV1',
 '6EF2-6EX1',
 '6EF2-6EZ1',
 '6EF3-6EX2',
 '6EF3-6EZ2',
 '6EF4-6EX3']

## Year codes are one digit and repeat across decades

`ESH6` is instrument 49705 in early 2016 and instrument 42140878 in 2025-26. A parser that only sees the string cannot order these; it needs the observation date. CL adds a twist: it is listed ten years out, so `CLM9` reappears as June 2029 within weeks of June 2019 expiring (instrument 323547 below).

In [5]:
esh6 = es[es["symbol"] == "ESH6"]
esh6.groupby("instrument_id").agg(
    first=("close", lambda s: s.index.min().date()),
    last=("close", lambda s: s.index.max().date()),
    rows=("close", "size"),
)

,first,last,rows
instrument_id,,,
49705,2016-01-03,2016-03-18,66
42140878,2025-01-31,2026-03-20,236


In [6]:
clm9 = raw["CL"][raw["CL"]["instrument_id"] == 323547]
clm9[["symbol", "close", "volume"]].iloc[[0, 1, 2, -1]]

,symbol,close,volume
ts_event,,,
2019-06-20 00:00:00+00:00,CLM9,54.10,8
2019-07-17 00:00:00+00:00,CLM9,54.55,8
2025-06-23 00:00:00+00:00,CLM9,64.22,2
2026-06-30 00:00:00+00:00,CLM9,64.69,46


## Bars are UTC days, not CME sessions

Globex opens Sunday 17:00 CT (22:00 or 23:00 UTC), so every week has a one-hour Sunday bar. Left in, the calendar has about 311 'days' a year and a weekly low-volume print that would spike any volume-normalised statistic.

In [7]:
days = pd.Series(es.index.normalize().unique())
print(days.dt.day_name().value_counts().to_dict())
print(f"distinct UTC days per year: {len(days) / 10.5:.0f}")
out = es[es["symbol"].str.fullmatch(outright_pattern("ES").pattern)]
sunday_share = out.loc[out.index.dayofweek == 6, 'volume'].sum() / out['volume'].sum()
print(f"Sunday share of outright volume: {sunday_share:.3%}")

{'Monday': 548, 'Tuesday': 548, 'Wednesday': 547, 'Thursday': 547, 'Sunday': 540, 'Friday': 536}
distinct UTC days per year: 311
Sunday share of outright volume: 0.306%


## Volume sanity

On a normal day the front contract dominates. Near a roll the calendar spread can out-trade the back month, and on some days (102 of them for ZB) it out-trades everything, so filtering to outrights before picking the leader is not optional.

In [8]:
es.loc["2024-03-01"].sort_values("volume", ascending=False)[["symbol", "close", "volume"]].head()

,symbol,close,volume
ts_event,,,
2024-03-01 00:00:00+00:00,ESH4,5138.75,1492386
2024-03-01 00:00:00+00:00,ESM4,5200.00,11740
2024-03-01 00:00:00+00:00,ESH4-ESM4,61.75,5739
2024-03-01 00:00:00+00:00,ESH4-ESU4,116.20,157
2024-03-01 00:00:00+00:00,ESU4,5255.75,131
